# Import

In [3]:
import pandas as pd
import numpy as np

# Constantes

In [7]:
# Processed Data Path
PATH_PROCESSED = "../data/processed/"

# Palpites
FILE_TIPS = "palpites__fg_processados.csv"
# Paises
FILE_PAISES = "apoio_paises.csv"

# ETL

## Leitura e Join com apoio

In [32]:
df_paises = pd.read_csv(PATH_PROCESSED + FILE_PAISES)

In [33]:
df_tips_table = pd.read_csv(PATH_PROCESSED + FILE_TIPS)
df_tips_table["nm_pais"] = df_tips_table["nm_time_casa"]

In [34]:
df_tips_table_2 = pd.merge(df_tips_table, df_paises, on='nm_pais', how='left')
df_tips_table_2.drop(["nm_pais"], axis=1, inplace=True)

In [ ]:
df_tips_table_3 = df_tips_table_2.dropna()
df_tips_table_3['id_pais'] = df_tips_table_3['id_pais'].astype(int)

## Gerar classificação

In [45]:
df = df_tips_table_3.copy()

# resultados por mandante
home = df.rename(columns={
    "nm_time_casa": "team",
    "nm_time_fora": "opp",
    "vl_time_casa": "gf",
    "vl_time_fora": "ga",
})

In [51]:
home["pts"] = np.select(
    [home["gf"] > home["ga"], home["gf"] == home["ga"]],
    [3, 1],
    default=0,
)

home["v"] = (home["gf"] > home["ga"]).astype(int)
home["e"] = (home["gf"] == home["ga"]).astype(int)
home["d"] = (home["gf"] < home["ga"]).astype(int)

In [49]:
home["v"] = (home["gf"] > home["ga"]).astype(int)
home["e"] = (home["gf"] == home["ga"]).astype(int)
home["d"] = (home["gf"] < home["ga"]).astype(int)

In [53]:
# resultados por visitante (espelha o jogo)
away = df.rename(columns={
    "nm_time_fora": "team",
    "nm_time_casa": "opp",
    "vl_time_fora": "gf",
    "vl_time_casa": "ga",
})

In [55]:
away["pts"] = np.select(
    [away["gf"] > away["ga"], away["gf"] == away["ga"]],
    [3, 1],
    default=0,
)

away["v"] = (away["gf"] > away["ga"]).astype(int)
away["e"] = (away["gf"] == away["ga"]).astype(int)
away["d"] = (away["gf"] < away["ga"]).astype(int)

In [62]:
# concatena e agrega
team_rows = pd.concat([home, away], ignore_index=True)

In [74]:
base_table = (
    team_rows
    .groupby(["nm_player", "nm_grpo", "team"], as_index=False)
    .agg(
        pts=("pts", "sum"),
        jogos=("team", "size"),
        v=("v", "sum"),
        e=("e", "sum"),
        d=("d", "sum"),
        gp=("gf", "sum"),
        gc=("ga", "sum"),
    )
)

base_table["sg"] = base_table["gp"] - base_table["gc"]

base_table

,nm_player,nm_grpo,team,pts,jogos,v,e,d,gp,gc,sg
0,ana nath,A,Coreia do Sul,1,3,0,1,2,3,7,-4
1,ana nath,A,Europa D,7,3,2,1,0,7,2,5
2,ana nath,A,México,6,3,2,0,1,9,5,4
3,ana nath,A,África do Sul,3,3,1,0,2,4,9,-5
4,ana nath,B,Canadá,7,3,2,1,0,7,3,4
...,...,...,...,...,...,...,...,...,...,...,...
187,washington,K,Uzbequistão,7,3,2,1,0,7,2,5
188,washington,L,Croácia,3,3,1,0,2,4,5,-1
189,washington,L,Gana,5,3,1,2,0,7,4,3
190,washington,L,Inglaterra,1,3,0,1,2,1,5,-4


## NAO FUNCIONANDO

In [70]:
def sort_group(part_df):
    # part_df: linhas de um participante+grupo, já com stats agregados
    jogos = df[
        (df["nm_player"] == part_df["nm_player"].iloc[0]) &
        (df["nm_grpo"] == part_df["nm_grpo"].iloc[0])
    ]
    # helper para head-to-head entre subconjunto de times empatados
    def head_to_head(tied_teams):
        filt = jogos[
            (jogos["nm_time_casa"].isin(tied_teams)) &
            (jogos["nm_time_fora"].isin(tied_teams))
        ]
        if filt.empty:
            return None
        h = filt.rename(columns={"nm_time_casa":"team","nm_time_fora":"opp","vl_casa":"gf","vl_fora":"ga"})
        h["pts"] = np.select([h["gf"]>h["ga"], h["gf"]==h["ga"]],[3,1],default=0)
        h["sg"] = h["gf"]-h["ga"]
        a = filt.rename(columns={"nm_time_fora":"team","nm_time_casa":"opp","vl_fora":"gf","vl_casa":"ga"})
        a["pts"] = np.select([a["gf"]>a["ga"], a["gf"]==a["ga"]],[3,1],default=0)
        a["sg"] = a["gf"]-a["ga"]
        hh = pd.concat([h,a])
        return (
            hh.groupby("team", as_index=False)
              .agg(pts=("pts","sum"), sg=("sg","sum"), gp=("gf","sum"))
        )

    # ordena com desempates iterativos
    part_df = part_df.copy()
    part_df["order"] = 0  # será sobrescrita
    # passo base: pontos gerais
    part_df = part_df.sort_values(["pts","sg","gp","team"], ascending=[False,False,False,True])

    i = 0
    while i < len(part_df):
        # encontra bloco empatado em pontos
        same_pts = part_df.iloc[i]["pts"]
        block = part_df.index[part_df["pts"] == same_pts].tolist()
        j = i
        while j < len(part_df) and part_df.iloc[j]["pts"] == same_pts:
            j += 1
        block_idx = part_df.index[i:j]

        # aplica head-to-head se mais de 1 time empatado
        if len(block_idx) > 1:
            tied_teams = part_df.loc[block_idx, "team"].tolist()
            hh = head_to_head(tied_teams)
            if hh is not None:
                # mescla stats de confronto direto
                merged = part_df.loc[block_idx].merge(hh, on="team", how="left", suffixes=("", "_hh"))
                merged = merged.sort_values(
                    ["pts_hh","sg_hh","gp_hh","sg","gp","team"],
                    ascending=[False,False,False,False,False,True]
                )
                part_df.loc[merged.index, "order"] = range(i+1, i+1+len(merged))
                part_df = pd.concat([part_df.drop(index=merged.index), merged]).sort_values("order").drop(columns="order")
            i = j
        else:
            i += 1

    # se ainda restarem empates exatos, fallback no sort final:
    return part_df.sort_values(["pts","sg","gp","team"], ascending=[False,False,False,True])


In [73]:
standings = (
    base_table
    .groupby(["nm_player", "nm_grpo"], group_keys=False)
    .apply(sort_group)
    .reset_index(drop=True)
)

# opcional: numerar posições por grupo
standings["pos"] = (
    standings.groupby(["nm_player","nm_grpo"])
             .cumcount() + 1
)

print(standings.head())


KeyError: 'gf'